# RQ1 — Statistical Tests: Structural Nature

This notebook runs the formal statistical tests supporting the RQ1 results:

1. **Wilson confidence intervals** for per-agent conflict rates (conflicting merges / total merges)
2. **Kruskal-Wallis** test for the number of conflict chunks per merge across agents
3. **Post-hoc Dunn test** (Bonferroni-corrected) for pairwise agent comparisons
4. **Cliff's delta** effect sizes for pairwise differences
5. **Kruskal-Wallis** tests for side-size distributions (`v1_loc`, `v2_loc`, `resolution_loc`) across agents

Required packages (beyond the project venv): `scipy`, `statsmodels`, `scikit_posthocs`.
Install with: `pip install scikit-posthocs`  (scipy and statsmodels are already in the venv).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / 'analysis' / 'common.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import scikit_posthocs as sp

from analysis.common import load_tables, build_chunk_frame, build_merge_frame

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Imports OK')

## 1. Load data

In [ ]:
tables = load_tables()
chunks = build_chunk_frame(tables)    # chunk-level frame with agent, language, etc.
merges = build_merge_frame(tables)    # merge-level frame

print(f'Chunks: {len(chunks):,}  |  Merges: {len(merges):,}')
print('Agents:', chunks['agent'].value_counts().to_dict())

## 2. Wilson CIs for per-agent conflict rates

For each agent: what fraction of its internal merge commits produce at least one conflict?
We use the Wilson (score) interval, which handles small counts better than the Normal approximation.

In [ ]:
# merge-level: flag whether a merge is conflicting
merge_conflict = (
    merges
    .assign(is_conflicting=merges['n_chunks'] > 0)  # adjust column name if needed
    .groupby('agent')
    .agg(n_total=('is_conflicting', 'count'),
         n_conflicting=('is_conflicting', 'sum'))
    .reset_index()
)

rows = []
for _, row in merge_conflict.iterrows():
    lo, hi = proportion_confint(row['n_conflicting'], row['n_total'],
                                 alpha=0.05, method='wilson')
    rows.append({
        'agent': row['agent'],
        'n_total': int(row['n_total']),
        'n_conflicting': int(row['n_conflicting']),
        'rate': row['n_conflicting'] / row['n_total'],
        'ci_lo': lo,
        'ci_hi': hi,
    })

ci_df = pd.DataFrame(rows).sort_values('rate', ascending=False)
ci_df['rate_pct'] = ci_df['rate'].map('{:.1%}'.format)
ci_df['ci'] = ci_df.apply(lambda r: f'[{r.ci_lo:.1%}, {r.ci_hi:.1%}]', axis=1)
print(ci_df[['agent', 'n_total', 'n_conflicting', 'rate_pct', 'ci']].to_string(index=False))

## 3. Kruskal-Wallis test: chunks per conflicting merge across agents

H₀: The distribution of the number of conflict chunks per merge commit is the same
across all five coding agents.

We restrict to conflicting merges only (n_chunks > 0) so the zero-inflation of
conflict-free merges does not dominate the test.

**Validity note:** Kruskal-Wallis assumes independence. Merges within the same
repository are correlated (same codebase, same contributor patterns), so the
effective sample size is smaller than the nominal count and p-values are
anti-conservative. Use Cliff's delta (§5) as the primary effect-size metric;
it is rank-based and does not depend on the distributional assumption.

In [ ]:
# Keep only conflicting merges
conflicting = merges[merges['n_chunks'] > 0].copy()  # adjust column name if needed

groups_chunks = [
    conflicting.loc[conflicting['agent'] == agent, 'n_chunks'].values
    for agent in conflicting['agent'].unique()
]
agent_labels = list(conflicting['agent'].unique())

kw_stat, kw_p = stats.kruskal(*groups_chunks)
print(f'Kruskal-Wallis H = {kw_stat:.3f},  p = {kw_p:.2e}')
if kw_p < 0.05:
    print('  → Reject H₀: at least one agent differs significantly.')
else:
    print('  → Fail to reject H₀.')

# Descriptive stats per agent
conflicting.groupby('agent')['n_chunks'].describe(percentiles=[.25,.50,.75,.90,.99]).round(1)

## 4. Post-hoc pairwise Dunn test (Bonferroni correction)

If Kruskal-Wallis is significant, identify which pairs of agents differ.

In [ ]:
dunn_result = sp.posthoc_dunn(
    conflicting, val_col='n_chunks', group_col='agent', p_adjust='bonferroni'
)
print('Dunn pairwise p-values (Bonferroni-corrected):')
print(dunn_result.round(4).to_string())
print('\nSignificant pairs (p < 0.05):')
agents = dunn_result.index.tolist()
for i, a1 in enumerate(agents):
    for a2 in agents[i+1:]:
        p = dunn_result.loc[a1, a2]
        if p < 0.05:
            print(f'  {a1} vs {a2}: p={p:.4f}')

## 5. Effect sizes: Cliff's delta for pairwise agent comparisons

Cliff's δ ∈ [−1, +1]. Interpretation thresholds (Romano et al.):
|δ| < 0.147 → negligible; < 0.330 → small; < 0.474 → medium; ≥ 0.474 → large.

In [ ]:
def cliffs_delta(x, y):
    """Non-parametric effect size: Cliff's delta."""
    x, y = np.asarray(x), np.asarray(y)
    m, n = len(x), len(y)
    dominance = sum((xi > yj) - (xi < yj) for xi in x for yj in y)
    return dominance / (m * n)

def delta_label(d):
    a = abs(d)
    if a < 0.147: return 'negligible'
    if a < 0.330: return 'small'
    if a < 0.474: return 'medium'
    return 'large'

print(f"{'Agent 1':<18} {'Agent 2':<18} {'Cliff\'s δ':>10} {'Effect':>12}")
print('-' * 62)
for i, a1 in enumerate(agent_labels):
    for a2 in agent_labels[i+1:]:
        x = conflicting.loc[conflicting['agent'] == a1, 'n_chunks'].values
        y = conflicting.loc[conflicting['agent'] == a2, 'n_chunks'].values
        d = cliffs_delta(x, y)
        print(f'{a1:<18} {a2:<18} {d:>10.3f} {delta_label(d):>12}')

## 6. Kruskal-Wallis tests for LOC distributions across agents

Repeat the test for `v1_loc`, `v2_loc`, `base_loc`, and `resolution_loc`.
H₀ for each: the LOC distribution is the same across all agents.

**Validity note:** same as §3 — within-repo clustering makes p-values
anti-conservative. Report medians and IQRs alongside the H statistic.
Cliff's delta (§5) remains the primary per-pair effect measure.

In [ ]:
loc_cols = ['v1_loc', 'v2_loc', 'base_loc', 'resolution_loc']

results = []
for col in loc_cols:
    if col not in chunks.columns:
        print(f'  Column {col!r} not found — skipping')
        continue
    groups = [
        chunks.loc[chunks['agent'] == agent, col].dropna().values
        for agent in chunks['agent'].unique()
    ]
    h, p = stats.kruskal(*groups)
    results.append({'metric': col, 'H': round(h, 2), 'p': p,
                    'significant': 'YES' if p < 0.05 else 'no'})

print(pd.DataFrame(results).to_string(index=False))

## 7. Kruskal-Wallis: chunks per merge across top languages

Same test as §3, stratified by programming language instead of agent.

In [ ]:
# Restrict to languages with at least 50 conflicting merges for stability
lang_counts = conflicting['language'].value_counts()
top_langs = lang_counts[lang_counts >= 50].index.tolist()

groups_lang = [
    conflicting.loc[conflicting['language'] == lang, 'n_chunks'].values
    for lang in top_langs
]
h_lang, p_lang = stats.kruskal(*groups_lang)
print(f'Kruskal-Wallis H = {h_lang:.3f},  p = {p_lang:.2e}  ({len(top_langs)} languages with ≥50 merges)')

# Medians per language
conflicting[conflicting['language'].isin(top_langs)].groupby('language')['n_chunks'].median().sort_values(ascending=False)

## 8. Summary table

Collect all test results for easy copy-paste into the paper.

In [ ]:
print('=== RQ1 STATISTICAL TEST SUMMARY ===')
print()
print('(1) Per-agent conflict rates — see ci_df above')
print(f'(2) Kruskal-Wallis (chunks per conflicting merge, across agents): H={kw_stat:.2f}, p={kw_p:.2e}')
print('(3) Post-hoc Dunn pairwise p-values (Bonferroni) — see dunn_result above')
print('(4) Cliff\'s delta effect sizes [PRIMARY EFFECT METRIC] — see table above')
print('(5) Kruskal-Wallis for LOC metrics — see results table above')
print(f'(6) Kruskal-Wallis (chunks per merge, across languages): H={h_lang:.2f}, p={p_lang:.2e}')
print()
print('VALIDITY NOTE: All KW tests assume independence. Within-repo clustering means')
print('p-values are anti-conservative (too small). Cliff\'s delta (rank-based, non-')
print('parametric) is the primary effect-size metric and is less affected by clustering.')
